In [9]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

In [ ]:
import torch
from datasets import load_dataset, Audio
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import os
import shutil
from transformers import TrainerCallback

import matplotlib.pyplot as plt

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load dataset from parquet folder
data_dir = "D:/vietbud500/viet_bud500/data"
dataset = load_dataset("parquet", data_dir=data_dir)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Load dataset
dataset = load_dataset("viet_bud500", split="train+validation+test")

Using device: cpu


Resolving data files:   0%|          | 0/105 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/91 [00:00<?, ?it/s]

In [12]:
# Preprocessing function
def preprocess(batch):
    audio = batch["audio"]["array"]
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

In [ ]:
# Apply preprocessing
processed_dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names)

# Data collator
data_collator = DataCollatorForSeq2Seq(processor.tokenizer, model=model, padding=True)

In [ ]:
# Custom callback to save and delete model checkpoints

class SaveAndDeleteCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.last_checkpoint = None

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        new_ckpt = os.path.join(self.output_dir, f"checkpoint-epoch-{epoch}")
        # Save model
        kwargs["model"].save_pretrained(new_ckpt)
        # Delete previous checkpoint
        if self.last_checkpoint and os.path.exists(self.last_checkpoint):
            shutil.rmtree(self.last_checkpoint)
        self.last_checkpoint = new_ckpt

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./whisper_checkpoints",
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

In [ ]:
# For logging loss
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append(logs["loss"])

In [ ]:
loss_logger = LossLoggerCallback()
save_and_delete = SaveAndDeleteCallback(training_args.output_dir)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    data_collator=data_collator,
    callbacks=[loss_logger, save_and_delete]
)

# Train
trainer.train()

In [ ]:
# Plot loss
plt.plot(loss_logger.losses)
plt.xlabel("Logging Step")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()